# 12 — Monitoring with Prometheus and Grafana
Runs in the **Jupyter container on node-serve-monitor**.

In [ ]:
import time
import random
import requests
import numpy as np
from concurrent.futures import ThreadPoolExecutor

In [ ]:
SERVING_URL = "http://fastapi_lgbm:8000"
print(requests.get(f"{SERVING_URL}/health").json())
print(requests.get(f"{SERVING_URL}/metrics").text[:500])

## Monitor operational metrics

Open Prometheus at `http://A.B.C.D:9090` in a browser.

Check **Status > Target Health** to confirm the FastAPI endpoint is being scraped.

Try the following PromQL queries:

Request rate:
```
rate(http_requests_total{handler="/queue"}[1m])
```

Average latency (ms):
```
1000 * rate(http_request_duration_seconds_sum{handler="/queue"}[1m]) / rate(http_request_duration_seconds_count{handler="/queue"}[1m])
```

p95 latency (ms):
```
1000 * histogram_quantile(0.95, rate(http_request_duration_seconds_bucket{handler="/queue"}[1m]))
```

In [ ]:
def make_payload(n_songs=10):
    return {
        "session_id": f"session_{random.randint(1, 100000)}",
        "user_features": {
            "user_skip_rate": round(random.random(), 4),
            "user_favorite_genre_encoded": random.randint(0, 20),
            "user_watch_time_avg": round(random.uniform(10, 240), 2),
        },
        "candidate_songs": [
            {
                "video_id": f"video_{i}",
                "release_year": random.randint(1970, 2026),
                "context_segment": random.randint(0, 10),
                "genre_encoded": random.randint(0, 20),
                "subgenre_encoded": random.randint(0, 3000),
            }
            for i in range(n_songs)
        ],
    }

In [ ]:
# Generate load
for _ in range(100):
    requests.post(f"{SERVING_URL}/queue", json=make_payload())

## Build Grafana dashboard

Open Grafana at `http://A.B.C.D:3000` in a browser. Sign in with `admin` / `admin`.

Add Prometheus data source:

1. Connections > Add new connection > Prometheus
2. Set URL to `http://prometheus:9090`
3. Save & Test

Create dashboard:

1. Dashboards > Create dashboard > Add visualization
2. Select Prometheus data source
3. Add panels for:
   - Average latency (ms)
   - p50 / p95 / p99 latency
   - Request rate
   - Error rate (status 4xx / 5xx)
4. Save dashboard as "SmartQueue Service Monitoring"

In [ ]:
# Ramp up / ramp down load
load_pattern = [1, 2, 4, 8, 4, 2, 1]
delay_between_steps = 30

def send_continuous(duration_sec):
    t_end = time.time() + duration_sec
    while time.time() < t_end:
        requests.post(f"{SERVING_URL}/queue", json=make_payload(), timeout=10)

for workers in load_pattern:
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(send_continuous, delay_between_steps) for _ in range(workers)]
        for f in futures:
            f.result()

## Create alert rule

In the Grafana dashboard, click ⫶ on the latency panel > More > New Alert Rule.

1. Name: "High p95 latency"
2. Query: `1000 * histogram_quantile(0.95, rate(http_request_duration_seconds_bucket{handler="/queue"}[1m]))`
3. Condition: above 500
4. Evaluation interval: 30s, pending period: 1m
5. Save

Generate high load to trigger the alert:

In [ ]:
# Generate burst load to trigger alert
def send_continuous_burst(duration_sec):
    t_end = time.time() + duration_sec
    while time.time() < t_end:
        requests.post(f"{SERVING_URL}/queue", json=make_payload(n_songs=50), timeout=30)

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(send_continuous_burst, 120) for _ in range(16)]
    for f in futures:
        f.result()

## Monitor model output

Create a panel showing average engagement score:

```
sum(rate(prediction_score_sum[1m])) / sum(rate(prediction_score_count[1m]))
```

And score histogram:

```
rate(prediction_score_bucket[1m])
```

In [ ]:
# Collect scores from responses
scores = []
for _ in range(50):
    r = requests.post(f"{SERVING_URL}/queue", json=make_payload())
    if r.status_code == 200:
        for song in r.json().get("ranked_songs", []):
            scores.append(song["engagement_probability"])

print(f"mean: {np.mean(scores):.3f}")
print(f"min: {np.min(scores):.3f}")
print(f"max: {np.max(scores):.3f}")

## Promotion and rollback triggers

Promote to Production if during 30-minute canary window:

- error rate <= 1%
- p95 latency <= 800 ms
- invalid score rate <= 0.1%

Rollback immediately if:

- error rate > 2% for 5+ minutes
- p95 latency > 1200 ms for 10+ minutes
- health endpoint fails 3 times consecutively

In [ ]:
# Quick operational check
N = 100
latencies = []
errors = 0

for _ in range(N):
    t0 = time.time()
    r = requests.post(f"{SERVING_URL}/queue", json=make_payload(), timeout=30)
    latencies.append((time.time() - t0) * 1000)
    if r.status_code >= 400:
        errors += 1

print(f"p50: {np.percentile(latencies, 50):.1f} ms")
print(f"p95: {np.percentile(latencies, 95):.1f} ms")
print(f"error rate: {errors / N * 100:.2f}%")